In [1]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import os
import cv2
from torchvision.transforms import ToTensor

class GalaxyDataset(Dataset):
    def __init__(self, image_dir, annotation_csv, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms

        # Load full annotations
        all_annotations = pd.read_csv(annotation_csv)

        valid_filenames = []
        for filename in all_annotations["filename"].unique():
            img_path = os.path.join(image_dir, filename)
            if cv2.imread(img_path) is not None:
                valid_filenames.append(filename)
            else:
                print(f"⚠️ Skipping unreadable image: {img_path}")

        # Filter annotations to valid images only
        self.annotations = all_annotations[all_annotations["filename"].isin(valid_filenames)].copy()
        self.image_filenames = self.annotations["filename"].unique()

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        filename = self.image_filenames[idx]
        img_path = os.path.join(self.image_dir, filename)

        image = cv2.imread(img_path)
        if image is None:
            raise RuntimeError(f"Image suddenly unreadable during fetch: {img_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = ToTensor()(image)

        # Get boxes and labels
        records = self.annotations[self.annotations["filename"] == filename]
        boxes = records[["xmin", "ymin", "xmax", "ymax"]].values
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((boxes.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        return image, target


In [15]:
import torchvision
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes):
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model


In [45]:
test_val = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/NewTrainingData',
    annotation_csv='/home/demo-user/Downloads/LoTSS files/new_real_annotations.csv'
)
print(f"Dataset size: {len(test_val)}")
print(f"First filename: {test_val.image_filenames[0]}")
print(f"First image path: {test_val.image_dir}/{test_val.image_filenames[0]}")

# Try loading first item
img, target = test_val[0]
print(f"Image shape: {img.shape}")
print(f"Boxes: {target['boxes']}")

Dataset size: 4351
First filename: test49_region_19_z0.10.jpg
First image path: /home/demo-user/Downloads/LoTSS files/NewTrainingData/test49_region_19_z0.10.jpg
Image shape: torch.Size([3, 512, 512])
Boxes: tensor([[179., 179., 333., 333.]])


In [49]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/GAN+REAL_v2',
    annotation_csv='/home/demo-user/Downloads/LoTSS files/combined_annotations_v2.csv'
)

val_dataset = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/NewTrainingData',
    annotation_csv='/home/demo-user/Downloads/LoTSS files/new_real_annotations.csv'
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

Training images: 8351
Validation images: 4351


In [61]:
import torch
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import os
import cv2
import pandas as pd
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor

# --- Dataset Class ---
class GalaxyDataset(Dataset):
    def __init__(self, image_dir, annotation_csv, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms

        all_annotations = pd.read_csv(annotation_csv)

        valid_filenames = []
        for filename in all_annotations["filename"].unique():
            img_path = os.path.join(image_dir, filename)
            if cv2.imread(img_path) is not None:
                valid_filenames.append(filename)
            else:
                print(f"⚠️ Skipping unreadable image: {img_path}")

        self.annotations = all_annotations[all_annotations["filename"].isin(valid_filenames)].copy()
        self.image_filenames = self.annotations["filename"].unique()

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        filename = self.image_filenames[idx]
        img_path = os.path.join(self.image_dir, filename)

        image = cv2.imread(img_path)
        if image is None:
            raise RuntimeError(f"Image suddenly unreadable during fetch: {img_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = ToTensor()(image)

        records = self.annotations[self.annotations["filename"] == filename]
        boxes = records[["xmin", "ymin", "xmax", "ymax"]].values
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((boxes.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        return image, target

# --- Model ---
def get_model(num_classes):
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# --- Collate ---
def collate_fn(batch):
    return tuple(zip(*batch))

# --- Datasets ---
train_dataset = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/GAN+REAL_v2',
    annotation_csv='/home/demo-user/Downloads/LoTSS files/combined_annotations_v2.csv'
)

val_dataset = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/NewTrainingData',
    annotation_csv='/home/demo-user/Downloads/LoTSS files/new_real_annotations.csv'
)

# --- DataLoaders ---
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

# --- Device ---
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# --- Model Init ---
model = get_model(num_classes=2)
model.to(device)

# --- Optimizer ---
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# --- Training Loop ---
num_epochs = 10

for epoch in range(num_epochs):

    # Training
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

    for images, targets in pbar:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        pbar.set_postfix(loss=epoch_loss / (pbar.n + 1))

    avg_train_loss = epoch_loss / len(train_loader)

    # Validation
    model.train()
    val_loss = 0.0

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            images = list(img.to(device) for img in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1} — Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# --- Save Model ---
torch.save(model.state_dict(), "/home/demo-user/Downloads/LoTSS files/faster_rcnn_galaxy_v3.pth")
print("Model saved.")

Training images: 8351
Validation images: 4351
Using device: cuda


Epoch 1/10 [Val]: 100%|█████████████████████| 1088/1088 [03:27<00:00,  5.25it/s]


Epoch 1 — Train Loss: 0.0330 | Val Loss: 0.0380


Epoch 2/10 [Val]: 100%|█████████████████████| 1088/1088 [03:25<00:00,  5.30it/s]


Epoch 2 — Train Loss: 0.0216 | Val Loss: 0.0310


...

Epoch 8 — Train Loss: 0.0068 | Val Loss: 0.0101


Epoch 9/10 [Val]: 100%|█████████████████████| 1088/1088 [03:16<00:00,  5.53it/s]


Epoch 9 — Train Loss: 0.0069 | Val Loss: 0.0105


Epoch 10/10 [Val]: 100%|████████████████████| 1088/1088 [03:17<00:00,  5.51it/s]


Epoch 10 — Train Loss: 0.0064 | Val Loss: 0.0115
Model saved.


In [13]:
torch.save(model.state_dict(), "/home/demo-user/Downloads/faster_rcnn_galaxy.pth")


In [59]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Wed Apr  8 16:47:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 ...    Off |   00000000:01:00.0  On |                  N/A |
| 18%   41C    P8             23W /  250W |    7731MiB /   8192MiB |     16%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----